# Testing App Functions

In [1]:
%load_ext autoreload
%autoreload 2
import custom_functions as fn



Top-Level Keys in FPATHS dict:
dict_keys(['data', 'images', 'metadata', 'eda', 'models', 'results', 'readme'])


In [2]:
ai_avatar ="🤖"
user_avatar="💬"
## Adding caching to reduce api usage
import os
from langchain.cache import InMemoryCache
# from langchain.document_loaders import CSVLoader
from langchain_community.document_loaders import CSVLoader
from langchain.globals import set_llm_cache
from langchain.memory import ChatMessageHistory, ConversationBufferMemory
from langchain.prompts import (
    ChatPromptTemplate, PromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
)
import streamlit as st
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma, FAISS
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.callbacks import StreamlitCallbackHandler
from langchain.callbacks import StdOutCallbackHandler
from langchain.agents import OpenAIFunctionsAgent, AgentExecutor
from langchain_openai.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage, AIMessage
from langchain.prompts import MessagesPlaceholder
from langchain.tools.retriever import create_retriever_tool
from langchain_community.document_loaders import CSVLoader

from langchain.agents import AgentExecutor, create_openai_tools_agent
# Memory: agent token buffer used in original example blog post
from langchain.agents.openai_functions_agent.agent_token_buffer_memory import AgentTokenBufferMemory
from langchain.memory import ConversationBufferMemory
# set_llm_cache(InMemoryCache())

from langchain import hub
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools.retriever import create_retriever_tool


In [3]:

import custom_functions as fn
FPATHS = fn.load_filepaths_json()
fpath_llm_csv = FPATHS['data']['app']['reviews-with-target-for-llm_csv']
fpath_db = FPATHS['data']['app']['vector-db_dir']
import os
if os.path.exists(fpath_db):
    print("Vector db file exists. Connecting to previous db.")
    retriever = fn.load_vector_database(fpath_db, fpath_llm_csv, use_previous=True, as_retriever=True)
else:
    print("Vector db file does not exist. Constructing vector db.")
    retriever = fn.load_vector_database(fpath_db, fpath_llm_csv, use_previous=False, as_retriever=True)



Top-Level Keys in FPATHS dict:
dict_keys(['data', 'images', 'metadata', 'eda', 'models', 'results', 'readme'])
Vector db file exists. Connecting to previous db.
Using previous vector db...


In [4]:
retriever.get_relevant_documents("Cook time")

[Document(page_content='review: Two Stars: Cooking process to long and complicated.\nstars: 2\ngroup: Low', metadata={'source': 'app-assets/reviews-for-llm.csv', 'row': 3751, 'reviewerID': 'AMSG9YQ8W6LLC'}),
 Document(page_content="review: Love These: If you want to improve the texture, rinse under cold water, blanch for one minute, then dry with a paper towel, then pan fry for ~5 minutes, then cook in your sauce for at least 2 minutes so they absorb the flavor. Also, I've begun to cut these before eating because they're pretty hard to cut once they're on the plate.\nstars: 5\ngroup: High", metadata={'source': 'app-assets/reviews-for-llm.csv', 'row': 4057, 'reviewerID': 'A1TBOU09U1Z3NF'}),
 Document(page_content='review: Follow the directions!: The trick to these is to prepare them exactly as described in the instructions. Boil for two minutes and then dry in a medium hot pan. That resulted in a a nice texture that was enjoyable to eat.\nstars: 4\ngroup:', metadata={'source': 'app-asse

In [6]:
agent = fn.get_agent(retriever=retriever)
agent

<generator object get_agent at 0x2970a65e0>

In [10]:
# agent.invoke({"input":"Cook time"})


In [7]:
# agent.memory

In [11]:
template = fn.get_template_string_reviews()
template

"You are a helpful data analyst for answering questions about what customers said about a specific  Amazon product using only content from use reviews. Assume all user questions are asking about the content in the user reviews. Note the product metadata is:\n```Product Info:\n\nTitle = Miracle Noodle Zero Carb, Gluten Free Shirataki Pasta, Spinach Angel Hair, 7-Ounce (Pack of 24)\n\nBrand = Miracle Noodle\n\nPrice = $59.76\n\nCategories = Grocery & Gourmet Food; Pasta & Noodles; Noodles; Shirataki\n\nProductID = B007JINB0W\n```\n\nUse the following pieces of context (user reviews) to answer the user's question by summarizing the reviews. \n            If you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{agent_scratchpad}\n\n"

In [13]:


## Make retreieval tool
# if retriever is None:
    # retriever  = load_vector_database( fpath_db,fpath_llm_csv, k=k, use_previous=True, as_retriever=True)#, use_previous=False)
tool = create_retriever_tool(
    retriever,
    "search_reviews",
    "Search Amazon custom reviews for relevant information."
)
tools = [tool]



In [14]:
verbose = True
temperature=0.1


# Create the chatprompttemplate
prompt_template = OpenAIFunctionsAgent.create_prompt(
    system_message=SystemMessage(template),
    extra_prompt_messages=[MessagesPlaceholder(variable_name="history")],
)

if verbose:
    print(prompt_template.messages)
    
llm = ChatOpenAI(temperature=temperature, api_key=os.getenv("OPENAI_API_KEY")) #streaming=True,
agent = create_openai_tools_agent(llm, tools, prompt_template)

## Creating streamlit-friendly memory for streaming
agent_executor = AgentExecutor(agent=agent, tools=tools,  verbose=True, #return_intermediate_steps=True,
                                memory=ConversationBufferMemory(memory_key="history",return_messages=True)
                                )

[SystemMessage(content="You are a helpful data analyst for answering questions about what customers said about a specific  Amazon product using only content from use reviews. Assume all user questions are asking about the content in the user reviews. Note the product metadata is:\n```Product Info:\n\nTitle = Miracle Noodle Zero Carb, Gluten Free Shirataki Pasta, Spinach Angel Hair, 7-Ounce (Pack of 24)\n\nBrand = Miracle Noodle\n\nPrice = $59.76\n\nCategories = Grocery & Gourmet Food; Pasta & Noodles; Noodles; Shirataki\n\nProductID = B007JINB0W\n```\n\nUse the following pieces of context (user reviews) to answer the user's question by summarizing the reviews. \n            If you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{agent_scratchpad}\n\n"), MessagesPlaceholder(variable_name='history'), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')), MessagesPlaceholder(variable_name='a

In [15]:
type(agent)

langchain_core.runnables.base.RunnableSequence

In [16]:
type(agent_executor)

langchain.agents.agent.AgentExecutor

In [7]:

def get_agent(retriever=None,fpath_db=None,
              fpath_llm_csv=None, k=8, temperature=0.1, verbose=False,
             template_string_func=None):
    if template_string_func is None:
        template_string_func=fn.get_template_string_reviews
    if fpath_db is None:
        FPATHS = fn.load_filepaths_json()
        fpath_db = FPATHS['data']['app']['vector-db_dir']

    if fpath_llm_csv is None:
        FPATHS = fn.load_filepaths_json()
        fpath_llm_csv = FPATHS['data']['app']['reviews-with-target-for-llm_csv']

    
    ## Make retreieval tool
    if retriever is None:
        retriever  = fn.load_vector_database( fpath_db,fpath_llm_csv, k=k, use_previous=True, as_retriever=True)#, use_previous=False)
    tool = create_retriever_tool(
        retriever,
        "search_reviews",
        "Search Amazon custom reviews for relevant information."
    )
    tools = [tool]
    
    
   ## Get template via function for template string
    template = template_string_func()

    # Create the chatprompttemplate
    prompt_template = OpenAIFunctionsAgent.create_prompt(
        system_message=SystemMessage(template),
        extra_prompt_messages=[MessagesPlaceholder(variable_name="history")],
    )
    
    if verbose:
        print(prompt_template.messages)
        
    llm = ChatOpenAI(temperature=temperature, api_key=os.getenv("OPENAI_API_KEY")) #streaming=True,
    agent = create_openai_tools_agent(llm, tools, prompt_template)
    
    ## Creating streamlit-friendly memory for streaming
    agent_executor = AgentExecutor(agent=agent, tools=tools,  verbose=True, #return_intermediate_steps=True,
                                   memory=ConversationBufferMemory(memory_key="history",return_messages=True)
                                   )
    return agent_executor
            
            
agent_executor = get_agent(retriever=retriever)#, fpath_db=fpath_db, fpath_llm_csv=fpath_llm_csv, k=8, temperature=0.1, verbose=True,

Top-Level Keys in FPATHS dict:
dict_keys(['data', 'images', 'metadata', 'eda', 'models', 'results', 'readme'])
Top-Level Keys in FPATHS dict:
dict_keys(['data', 'images', 'metadata', 'eda', 'models', 'results', 'readme'])
Top-Level Keys in FPATHS dict:
dict_keys(['data', 'images', 'metadata', 'eda', 'models', 'results', 'readme'])


In [8]:
agent_executor

AgentExecutor(memory=ConversationBufferMemory(return_messages=True), verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'history', 'input'], input_types={'history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]], 'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, messages=[SystemMessage(content="You are a hel

In [10]:
agent_executor = fn.get_agent(retriever=retriever)#, fpath_db=fpath_db, fpath_llm_csv=fpath_llm_csv, k=8, temperature=0.1, verbose=True,
agent_executor

Top-Level Keys in FPATHS dict:
dict_keys(['data', 'images', 'metadata', 'eda', 'models', 'results', 'readme'])


<generator object get_agent at 0x299da2c00>